# Mining Frequent Patterns — Progressive In-Class Practice

**CSCI 480 / CIS 680 — Advanced Databases**

**Student version — complete the `TODO` portions as you progress.** Do not try to finish the notebook all at once. Run one cell, inspect its output, answer the associated question, and then continue.

**Learning goals:** transform shopping baskets into a Boolean table, understand support, use FP-growth to find frequent itemsets, generate association rules, interpret confidence and lift, and apply the workflow to AdventureWorks transaction data.

**Pacing:** Part 1 is suitable for an approximately 35–45 minute guided class activity; Part 2 extends the same process to SQL data and may require another class period.

Adapted from FAU-CS6, *KDD — Knowledge Discovery in Databases*, Exercise 1: Mining Frequent Patterns.

## Setup

Run the import cell. If `mlxtend` is missing, uncomment the installation command and execute it, then rerun the imports.

In [ ]:
# Uncomment only if needed:
# %pip install pandas mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

## Part 1 — Five small transactions

The five baskets are the same as in the lecture example. Study the table before writing code.

| TID | Items |
|---|---|
| 10 | Beer, Nuts, Diapers |
| 20 | Beer, Coffee, Diapers |
| 30 | Beer, Diapers, Eggs |
| 40 | Nuts, Eggs, Milk |
| 50 | Nuts, Coffee, Diapers, Eggs, Milk |

In [ ]:
# Working example: the transaction data are provided.
dataset = [
    {"Beer", "Nuts", "Diapers"},
    {"Beer", "Coffee", "Diapers"},
    {"Beer", "Diapers", "Eggs"},
    {"Nuts", "Eggs", "Milk"},
    {"Nuts", "Coffee", "Diapers", "Eggs", "Milk"},
]
print("Number of transactions:", len(dataset))
print("First basket:", sorted(dataset[0]))

### Checkpoint 1 — Understand the raw transactions

**Question:** How many transactions contain `Diapers`? How many contain both `Beer` and `Diapers`?

Complete the condition below. Recall that an item is **in** a basket if the basket contains it.

In [ ]:
# TODO: Count transactions containing Diapers.
diaper_count = sum(1 for basket in dataset if "_____" in basket)
print("Diaper transactions:", diaper_count)

# TODO: Count baskets that contain BOTH items.
beer_diaper_count = sum(
    1 for basket in dataset
    if "Beer" in basket and "_____" in basket
)
print("Beer and Diapers transactions:", beer_diaper_count)

### Checkpoint 2 — One-hot encoding

`fpgrowth()` needs one Boolean column per product, with one row per transaction.

**Step A (working example):** create and fit a `TransactionEncoder`. **Step B (your task):** transform the transactions and create a DataFrame. The encoder's `columns_` attribute gives the product names.

In [ ]:
# Step A: this part is provided.
encoder = TransactionEncoder()
encoder.fit(dataset)
print("Columns:", list(encoder.columns_))

# Step B: fill in the two blanks.
encoded_array = encoder.transform(_____)
preprocessed_dataset = pd.DataFrame(
    encoded_array,
    columns=encoder._____
)
display(preprocessed_dataset)

**Check:** The result should have **5 rows** and **5 Boolean product columns**. The row order matches the five baskets; the column order may be alphabetical.

In [ ]:
# Checkpoint: the table shape should be (5, 5).
print("Shape:", preprocessed_dataset.shape)

### Checkpoint 3 — Relative support (calculate before mining)

For an itemset X:

**support(X) = number of baskets containing X / total number of baskets**

**Question:** What is the support of `{Beer, Diapers}`? First write your prediction, then complete the calculation.

In [ ]:
# TODO: Use the count computed in Checkpoint 1.
support_beer_diapers = _____ / len(dataset)
print("Support of {Beer, Diapers}:", support_beer_diapers)

### Checkpoint 4 — Discover frequent itemsets using FP-growth

An itemset is frequent if its support is at least the threshold. A threshold of **0.4** means at least two out of five transactions.

Complete the missing argument, then run the cell.

In [ ]:
# TODO: Fill in the support threshold.
frequent_itemsets = fpgrowth(
    preprocessed_dataset,
    min_support=_____,
    use_colnames=True
)
display(frequent_itemsets.sort_values("support", ascending=False))

**Questions:** Find `{Beer, Diapers}` in the output. Does the support agree with your manual calculation? Which itemsets would disappear if you raised `min_support` from 0.4 to 0.6?

In [ ]:
# Experiment: change the threshold, then compare results.
stricter_itemsets = fpgrowth(
    preprocessed_dataset,
    min_support=_____,  # TODO: use 0.6
    use_colnames=True
)
display(stricter_itemsets.sort_values("support", ascending=False))

### Checkpoint 5 — Association rules

A rule `A → B` means transactions containing A tend to contain B. **Support** measures how often A and B occur together; **confidence** measures how often B occurs among transactions containing A.

Complete the two rule-generation arguments. For newer `mlxtend` versions, `num_itemsets` is also provided.

In [ ]:
# TODO: use the frequent itemsets from Checkpoint 4.
frequent_patterns = association_rules(
    frequent_itemsets,
    metric="_____",       # Hint: support
    min_threshold=_____,    # Hint: 0.4
    num_itemsets=len(dataset)
)
columns_to_show = ["antecedents", "consequents", "support", "confidence", "lift"]
display(frequent_patterns[columns_to_show])

### Checkpoint 6 — Interpret one rule

Use the results to identify the rule `{Beer} → {Diapers}`. Manually compute:

- `support({Beer, Diapers}) = (baskets with both) / 5`
- `confidence(Beer → Diapers) = (baskets with both) / (baskets with Beer)`

Then interpret **lift**: a value greater than 1 indicates a positive association relative to independent occurrence. Explain why high support does not necessarily imply high lift.

In [ ]:
# TODO: calculate confidence using your counts.
beer_count = sum(1 for basket in dataset if "Beer" in basket)
confidence_beer_to_diapers = _____ / beer_count
print("Confidence, Beer -> Diapers:", confidence_beer_to_diapers)

## Part 2 — AdventureWorks (SQLite)

Apply the same steps to an actual relational dataset. SQL retrieves and groups transaction records; Python performs the pattern mining.

**Codespaces setup:** A new Codespace installs `data/adventure-works.db` automatically. If missing or invalid, run `bash .devcontainer/setup.sh` in the terminal, then rerun the following cell.

In [ ]:
# Provided database connection: do not change unless your setup differs.
from pathlib import Path
import sqlite3

database_path = Path("data/adventure-works.db")
if not database_path.is_file():
    raise FileNotFoundError(
        "AdventureWorks database is missing. Run: bash .devcontainer/setup.sh"
    )
with database_path.open("rb") as database_file:
    if database_file.read(16) != b"SQLite format 3\x00":
        raise RuntimeError(
            "Not a valid SQLite database. Run: bash .devcontainer/setup.sh"
        )
connection = sqlite3.connect(database_path)
print(f"Connected to AdventureWorks SQLite: {database_path} "
      f"({database_path.stat().st_size / (1024 * 1024):.1f} MiB)")

### Checkpoint 7 — Explore the SQL tables

Run the provided queries, inspect sample rows, and identify which columns represent **one record**, **one order**, and **one product**.

In [ ]:
# Working example: retrieve the relevant columns first.
transaction_history_df = pd.read_sql_query(
    "SELECT * FROM TransactionHistory;", connection
)
product_df = pd.read_sql_query(
    "SELECT * FROM Product;", connection
)

print("TransactionHistory:", transaction_history_df.shape)
display(transaction_history_df.head(5))
print("Product:", product_df.shape)
display(product_df[["ProductID", "Name"]].head(5))

### Checkpoint 8 — Which identifier defines a basket?

`TransactionID` identifies a transaction-history row; `ReferenceOrderID` can identify a group of products in the same order. Compare the provided query to the incomplete second query.

**Question:** Why would grouping by `TransactionID` make it difficult to find items bought together?

In [ ]:
# Step A: provided query — multiple products per TransactionID?
by_transaction = pd.read_sql_query("""
SELECT TransactionID, COUNT(DISTINCT ProductID) AS product_count
FROM TransactionHistory
GROUP BY TransactionID
HAVING COUNT(DISTINCT ProductID) > 1
LIMIT 10;
""", connection)
print("Multiple products per TransactionID:")
display(by_transaction)

# Step B: complete the order-level query.
by_order = pd.read_sql_query("""
SELECT _____, COUNT(DISTINCT ProductID) AS product_count
FROM TransactionHistory
GROUP BY _____
HAVING COUNT(DISTINCT ProductID) > 1
LIMIT 10;
""", connection)
print("Multiple products per ReferenceOrderID:")
display(by_order)

### Checkpoint 9 — Convert relational rows into baskets

**Step A:** group `ProductID` by `ReferenceOrderID` (one list of unique products per order).

**Step B:** reuse `TransactionEncoder` from Part 1. Complete each step separately and display the number of baskets before continuing.

In [ ]:
# TODO Step A: fill the grouping key and product column.
products_per_order = (
    transaction_history_df
    .groupby("_____")["_____"]
    .apply(lambda values: list(set(values)))
)
print("Number of orders:", len(products_per_order))
display(products_per_order.head())

In [ ]:
# TODO Step B: encode order-level baskets.
adventure_encoder = TransactionEncoder()
adventure_array = adventure_encoder.fit(products_per_order).transform(_____)
adventure_baskets = pd.DataFrame(
    adventure_array,
    columns=adventure_encoder._____
)
print("Basket table shape:", adventure_baskets.shape)
display(adventure_baskets.head())

### Checkpoint 10 — Find frequent product sets

Unlike the five-basket example, this real database can be sparse. Start with **minimum itemset support = 0.01**. Complete the function call.

In [ ]:
# TODO: apply FP-growth to the AdventureWorks Boolean table.
adventure_itemsets = fpgrowth(
    _____,
    min_support=_____,
    use_colnames=True
)
print("Frequent itemsets:", len(adventure_itemsets))
display(adventure_itemsets.sort_values("support", ascending=False).head(10))

### Checkpoint 11 — Generate and rank association rules

Generate rules using **minimum rule support = 0.02**, then display the ten highest-support rules. Reuse the technique from Part 1.

In [ ]:
# TODO: fill in the frequent-itemset variable and threshold.
adventure_rules = association_rules(
    _____,
    metric="support",
    min_threshold=_____,
    num_itemsets=len(products_per_order)
)

# TODO: sort rules by support, descending, then take ten.
top_rules = adventure_rules.sort_values(
    "_____", ascending=False
).head(_____)

display(top_rules[["antecedents", "consequents", "support", "confidence", "lift"]])

### Checkpoint 12 — Translate product IDs into names

Numeric IDs are difficult to interpret. Build a product-name lookup from `product_df`, then complete the function to translate each itemset.

In [ ]:
# Provided product-name lookup.
product_names = product_df.set_index("ProductID")["Name"].to_dict()

# TODO: replace one blank to look up each ProductID.
def describe_products(itemset):
    return ", ".join(str(product_names.get(pid, _____)) for pid in sorted(itemset))

# TODO: use describe_products on both sides of each rule.
readable_rules = top_rules.copy()
readable_rules["antecedent_names"] = readable_rules["antecedents"].apply(_____)
readable_rules["consequent_names"] = readable_rules["consequents"].apply(_____)
display(readable_rules[["antecedent_names", "consequent_names", "support", "confidence", "lift"]])

### Reflection — Write your answers

1. Why must database rows be grouped into orders before mining?
2. What is the difference between `TransactionID` and `ReferenceOrderID`?
3. How does increasing minimum support affect the number of frequent itemsets?
4. Compare support, confidence, and lift for one AdventureWorks rule.
5. Why are product names more useful than ProductIDs when reporting results?
6. Does a rule with high support necessarily have high lift? Explain.

**Source:** FAU-CS6/KDD, *Mining Frequent Patterns*, Task 4, and [FAU-CS6/KDD-Databases](https://github.com/FAU-CS6/KDD-Databases). This guided version uses FP-growth only; it omits Apriori and advanced interestingness-measure exercises.

In [ ]:
# Optional: close the database connection when finished.
# connection.close()